# GloVe (mean pooling, concatenated) + bigger MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:** Same framework as `w2v_mean_mlp.ipynb`, three changes:

- glove vocabulary (`glove-wiki-gigaword-300`, uncased) -> tokens are lowercased, much lower OOV
- the four sentence vectors are concatenated in fixed order O1, O2, H1, H2 instead of averaged -> 1200 dims, swapping H1 and H2 changes the input
- one more linear layer + ReLU in the MLP

In [1]:
%pip install -q datasets gensim lightning nltk python-dotenv wandb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))  # notebooks/project_1 -> shared pipeline package

import lightning as L
import torch
import torch.nn as nn

from pipeline import tracking
from pipeline.data import DATASET, TEXT_FIELDS, load_splits
from pipeline.features.static_embeddings import MeanEmbedder, build_features, combine_concat, load_vectors
from pipeline.loaders import make_loader
from pipeline.preprocessing import tokenize_split
from pipeline.preprocessing import nltk_alnum_lower as preprocessing  # swap the preprocessing variant here
from pipeline.training import evaluate_best, fit, training_summary, weights_dir

True

In [3]:
tracking.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
RUN_NAME = "glove-concatenate-bigger-mlp"  # W&B run + artifact name, weights in models/project_1/glove_concatenate_bigger_mlp/weights

config = {
    "dataset": DATASET,
    **preprocessing.CONFIG,  # tokenizer, lowercase, remove_punctuation, remove_stopwords, normalization, truncation
    "embedding": "glove-wiki-gigaword-300",
    "sentence_representation": "mean",
    "example_representation": "concat",  # O1, O2, H1, H2
    "classifier": "mlp_2_hidden",
    "hidden_dim": 256,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}

run = tracking.init_run(RUN_NAME, config)

In [5]:
train, valid, test = load_splits()

train: 16801, valid: 1000, test: 1532


In [8]:
train_tokens = tokenize_split(train, preprocessing.preprocess)
valid_tokens = tokenize_split(valid, preprocessing.preprocess)
test_tokens = tokenize_split(test, preprocessing.preprocess)

Map:   0%|          | 0/16801 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1532 [00:00<?, ? examples/s]

In [9]:
wv = load_vectors(config["embedding"])

vocabulary: 400000 words, 300 dimensions


In [12]:
embed = MeanEmbedder(wv)

X_train, y_train = build_features(train_tokens, embed, combine_concat)
X_valid, y_valid = build_features(valid_tokens, embed, combine_concat)
X_test, y_test = build_features(test_tokens, embed, combine_concat)

### DataLoaders - Takes care of Training Batching and Loading

In [13]:
L.seed_everything(config["seed"])

train_loader = make_loader(X_train, y_train, config["batch_size"], shuffle=True)
valid_loader = make_loader(X_valid, y_valid, config["batch_size"], shuffle=False)
test_loader = make_loader(X_test, y_test, config["batch_size"], shuffle=False)

Seed set to 42


### LightningModule Model (Derived form libraries.pdf using coding agents in combination with Lightning Documentation)

In [14]:
class MLP(L.LightningModule): # @todo lightning implementation fine here? any alternatives

    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate):
        super().__init__()
        self.save_hyperparameters()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def reset_parameters(self):
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                layer.reset_parameters()

    def forward(self, x):
        return self.layers(x)

     # @todo investigate and evaluate / tune learning rate
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

    def shared_step(self, batch, split): # @todo this has to be explainable in detailed since its activation loss etc.
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    # @todo check that against ML libraries
    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

In [15]:
model = MLP(
    input_dim=len(TEXT_FIELDS) * wv.vector_size,
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    learning_rate=config["learning_rate"],
)
model.reset_parameters()

trainer = fit(model, train_loader, valid_loader, weights_dir(RUN_NAME), max_epochs=config["max_epochs"], patience=config["patience"])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layers  │ Sequential       │  373 K │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 373 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 373 K                                                                                                
Total estimated model params size (MB): 1.495                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Metric loss/valid improved. New best score: 0.672
Metric loss/valid improved by 0.007 >= min_delta = 0.0. New best score: 0.665
Metric loss/valid improved by 0.004 >= min_delta = 0.0. New best score: 0.661
Metric loss/valid improved by 0.008 >= min_delta = 0.0. New best score: 0.653
Monitored metric loss/valid did not improve in the last 5 records. Best score: 0.653. Signaling Trainer to stop.


### Best Checkpoint + Upload Artifacts to Weights and Biases

In [16]:
evaluate_best(trainer, model, valid_loader, test_loader)

run.summary.update({
    **training_summary(trainer, y_test),  # accuracy/majority, best/loss_valid, best/checkpoint, early_stopping/stopped_epoch
    **embed.summary(),  # oov/tokens, oov/occurrences, oov/rate, oov/unique_types, oov/empty_sentences
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_bigger_mlp/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_bigger_mlp/weights/best-epoch=3.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │    0.5979999899864197     │
│        loss/valid         │    0.6527876257896423     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_bigger_mlp/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_bigger_mlp/weights/best-epoch=3.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.5169712901115417     │
│         loss/test         │    0.7795184850692749     │
└───────────────────────────┴───────────────────────────┘

In [17]:
tracking.log_best_checkpoint(run, RUN_NAME, trainer.checkpoint_callback)
run.finish()

accuracy/test,▁
accuracy/train,▁▃▄▄▅▅▆▇█
accuracy/valid,▁▆▇▇▇█▆▆▁▇
epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
loss/test,▁
loss/train,█▇▆▆▅▅▄▂▁
loss/valid,▂▂▁▁▂▃▃▅█▁
trainer/global_step,▁▁▂▂▃▃▄▄▄▄▅▅▆▆▇▇████
accuracy/majority,0.50979
accuracy/test,0.51697
accuracy/train,0.76359
